# AE3 - Transformación y enriquecimiento de datos (Actividad 4)
## Proyecto integrador: cancelación de clientes en una empresa de telecomunicaciones

**Autor:** Diego Mauricio Bedoya Valencia - Institución Universitaria Digital de Antioquia

Flujo: `dataset AE2 (Hugging Face)` → `transformación` → `enriquecimiento` → `respuestas a las preguntas` → `tablas .csv + resumen .txt`

## 1. Necesidad, preguntas y variables

**Necesidad (AE2):** el equipo de retención necesita saber **en qué grupos de clientes se concentra la cancelación** para priorizar sus campañas.

**Ajuste justificado:** en AE2 se planteó una sola pregunta general. Para responderla con tablas concretas se divide en 6 subpreguntas (una por dimensión mencionada) y una línea base. El alcance, el dataset y la unidad de análisis (el cliente) no cambian.

| # | Pregunta | Variables | Resultado esperado |
|---|---|---|---|
| P0 | ¿Cuál es la tasa global de cancelación? (línea base) | `cancelo`, `cargo_mensual` | Clientes, % y cargo mensual por estado |
| P1 | ¿Cómo varía la tasa según las condiciones de la cuenta? | `tipo_contrato`, `metodo_pago`, `factura_electronica`, `pago_automatico`* | Tasa por categoría |
| P2 | ¿Cómo varía la tasa según la antigüedad? | `antiguedad_meses` → `rango_antiguedad`* | Tasa por rango de meses |
| P3 | ¿Cómo varía la tasa según los servicios contratados? | `servicio_internet`, `tipo_servicio`*, 6 servicios adicionales, `n_servicios_adicionales`*, `lineas_multiples` | Tasa por servicio |
| P4 | ¿Cómo varía la tasa según el perfil demográfico? | `genero`, `adulto_mayor`, `tiene_pareja`, `tiene_dependientes`, `tiene_familia`* | Tasa por perfil |
| P5 | ¿Cómo se diferencian los cargos entre quienes cancelan y quienes no? | `cargo_mensual`, `cargo_total`, `rango_cargo_mensual`* | Estadísticas por grupo y tasa por rango |
| P6 | ¿Qué segmentos combinados tienen las tasas más altas y cuánto ingreso concentran? | `tipo_contrato` × `servicio_internet` × `rango_antiguedad`, `cargo_mensual_perdido`* | Ranking de segmentos |

\* Variable creada en el enriquecimiento.

**Limitaciones que se mantienen:** datos ficticios; no hay fechas (no se analiza evolución), costos ni motivos de cancelación; la moneda no está documentada (se usa *unidades monetarias, u.m.*). Los resultados son descriptivos, no causales.

## 2. Preparación del entorno y carga del dataset

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
import pandas as pd
from datasets import disable_progress_bars
from huggingface_hub.utils import logging as hf_logging

disable_progress_bars()
hf_logging.set_verbosity_error()

# Raiz del repositorio (carpeta que contiene AE2 y AE3), con rutas relativas
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AE3" / "src").exists())
AE3 = RAIZ / "AE3"
sys.path[:0] = [str(RAIZ / "AE2" / "src"), str(AE3 / "src")]

from hf_dataset_loader import HFDatasetLoader          # clase de AE2 (se reutiliza)
from preparacion_Actividad4 import (perfil_calidad, transformar, controles_calidad,
                                    enriquecer, tabla_tasa, validar_tabla,
                                    ResumenDescriptivo, ADICIONALES)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print("Raiz del proyecto:", RAIZ.name, "| pandas", pd.__version__)

Raiz del proyecto: proyecto_integrador | pandas 3.1.1


In [2]:
DATASET_ID = "scikit-learn/churn-prediction"
REVISION = "aa09900373d90780ee70d27571775aff0e51569c"   # misma version de AE2

df_original = HFDatasetLoader(DATASET_ID, split="train", revision=REVISION).load()
df_original_copia = df_original.copy()   # version sin modificar para comparar
print("Dataset original:", df_original.shape)
df_original.head(3)

Dataset original: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 3. Transformación y validación

Reglas aplicadas (función `transformar` en `AE3/src/preparacion_Actividad4.py`):

1. **Espacios sobrantes** en textos: se eliminan para que los vacíos y categorías se detecten bien.
2. **`TotalCharges` texto → número.** Tiene 11 textos vacíos; todos son clientes con `tenure = 0` (recién vinculados, aún sin facturación acumulada). **Se reemplazan por 0** y no se eliminan: eliminarlos sacaría del análisis a clientes nuevos reales; el valor 0 es coherente con `cargo_mensual × antigüedad = 0`. El código verifica que ningún vacío tenga antigüedad mayor que 0.
3. **`SeniorCitizen` 0/1 → No/Sí**, para tratarlo como categoría igual que las demás variables de perfil.
4. **Normalización de categorías y nombres:** valores y columnas en español (ej. `Month-to-month` → `Mes a mes`, `Churn` → `cancelo`). En `servicio_internet`, el valor `No` se unifica como `Sin internet`, igual que en los servicios adicionales. Si aparece una categoría sin regla, el código se detiene.
5. **Tipos:** las categóricas pasan a `category` (con orden en contrato y rangos).

**Selección de columnas y registros:** se conservan los **7.043 clientes** y las **21 columnas**, porque todas se usan en alguna pregunta (`id_cliente` solo para contar). No hay registros fuera del alcance.

In [3]:
df_transformado, registro_cambios = transformar(df_original)
registro_cambios

,transformacion,registros_o_columnas_afectados
0,Quitar espacios en columnas de texto,11
1,TotalCharges a numero; vacios (tenure = 0) ree...,11
2,SeniorCitizen 0/1 convertido a No/Sí,7043
3,Categorias traducidas al espanol (17 columnas),7043
4,Columnas renombradas a espanol snake_case,21
5,Columnas categoricas convertidas a tipo category,17


### 3.1 Controles de calidad sobre el DataFrame transformado

In [4]:
controles = controles_calidad(df_transformado)
controles

,control,registros_con_problema,resultado
0,id_cliente duplicado,0,OK
1,filas duplicadas completas,0,OK
2,filas iguales salvo id_cliente (clientes disti...,22,Informativo
3,valores nulos (total),0,OK
4,antiguedad_meses fuera de 0-72,0,OK
5,cargo_mensual <= 0,0,OK
6,cargo_total < 0,0,OK
7,lineas_multiples incoherente con servicio_tele...,0,OK
8,servicios adicionales incoherentes con servici...,0,OK
9,cargo_total / (cargo_mensual x antiguedad) fue...,0,OK


**Resultado de los controles:**
- No hay `id_cliente` duplicados ni filas duplicadas completas.
- Hay **22 filas iguales en todo salvo el `id_cliente`**: son clientes distintos con el mismo perfil (variables con pocas categorías hacen esto posible). **Se conservan**, porque eliminarlas reduciría clientes reales del conteo.
- Sin nulos tras el tratamiento, sin valores fuera de rango y sin incoherencias entre servicio base y servicios dependientes.
- `cargo_total` es coherente con `cargo_mensual × antigüedad` (razón entre 0,5 y 2 en todos los casos); las pequeñas diferencias se explican por cambios de tarifa, por eso no se corrige.

### 3.2 Comparación antes y después

In [5]:
antes, despues = perfil_calidad(df_original_copia), perfil_calidad(df_transformado)
despues.index = list(antes.index)  # alinear por nombre original
comparacion = pd.DataFrame({
    "columna_nueva": df_transformado.columns,
    "tipo_antes": antes["tipo"], "tipo_despues": despues["tipo"],
    "nulos_antes": antes["nulos"], "vacios_antes": antes["textos_vacios"],
    "nulos_despues": despues["nulos"], "unicos_antes": antes["valores_unicos"],
    "unicos_despues": despues["valores_unicos"],
})
comparacion.index.name = "columna_original"
print(f"Filas:    antes {df_original_copia.shape[0]}  -> despues {df_transformado.shape[0]}")
print(f"Columnas: antes {df_original_copia.shape[1]}  -> despues {df_transformado.shape[1]}")
print(f"Nulos + vacios: antes {int(antes['nulos'].sum() + antes['textos_vacios'].sum())} -> despues {int(despues['nulos'].sum())}")
comparacion

Filas:    antes 7043  -> despues 7043
Columnas: antes 21  -> despues 21
Nulos + vacios: antes 11 -> despues 0


,columna_nueva,tipo_antes,tipo_despues,nulos_antes,vacios_antes,nulos_despues,unicos_antes,unicos_despues
columna_original,,,,,,,,
customerID,id_cliente,str,string,0,0,0,7043,7043
gender,genero,str,category,0,0,0,2,2
SeniorCitizen,adulto_mayor,int64,category,0,0,0,2,2
Partner,tiene_pareja,str,category,0,0,0,2,2
Dependents,tiene_dependientes,str,category,0,0,0,2,2
tenure,antiguedad_meses,int64,int64,0,0,0,73,73
PhoneService,servicio_telefonico,str,category,0,0,0,2,2
MultipleLines,lineas_multiples,str,category,0,0,0,3,3
InternetService,servicio_internet,str,category,0,0,0,3,3


In [6]:
# El original se conserva sin cambios
assert df_original.equals(df_original_copia)
assert len(df_transformado) == len(df_original)
print("OK: el original no se modifico y no se perdieron registros.")

OK: el original no se modifico y no se perdieron registros.


## 4. Enriquecimiento

No se combina otra tabla: las variables del dataset son suficientes para la necesidad (por eso no hay riesgo de multiplicar registros por una unión). Se crean 9 variables:

| Variable | Regla / fórmula | Utilidad |
|---|---|---|
| `cancelo_flag` | 1 si `cancelo = Sí`, 0 si no | Calcular tasas con suma y promedio (P0-P6) |
| `rango_antiguedad` | `antiguedad_meses` en 0-12, 13-24, 25-48, 49-72 meses (definido en AE2) | P2, P6 |
| `rango_cargo_mensual` | `cargo_mensual`: Bajo <35, Medio 35-70, Alto 70-90, Muy alto ≥90 (aprox. cuartiles: 35,5 / 70,35 / 89,85) | P5 |
| `tipo_servicio` | Solo teléfono / Solo internet / Teléfono e internet | P3 |
| `n_servicios_adicionales` | Conteo de "Sí" en los 6 servicios adicionales (0 a 6) | P3 |
| `rango_servicios_adicionales` | 0, 1-2, 3-4, 5-6 servicios | P3 |
| `pago_automatico` | Sí si el método de pago es automático (transferencia o tarjeta) | P1 |
| `tiene_familia` | Sí si tiene pareja o dependientes | P4 |
| `cargo_mensual_perdido` | `cargo_mensual × cancelo_flag` (u.m./mes) | Ingreso mensual que deja de percibirse (P1-P6) |

In [7]:
df_final = enriquecer(df_transformado)
nuevas = [c for c in df_final.columns if c not in df_transformado.columns]
print("Filas:", len(df_transformado), "->", len(df_final), "| Columnas:", df_transformado.shape[1], "->", df_final.shape[1])
print("Variables nuevas:", nuevas)
df_final[["antiguedad_meses", "rango_antiguedad", "cargo_mensual", "rango_cargo_mensual",
          "tipo_servicio", "n_servicios_adicionales", "pago_automatico", "cancelo",
          "cancelo_flag", "cargo_mensual_perdido"]].head()

Filas: 7043 -> 7043 | Columnas: 21 -> 30
Variables nuevas: ['cancelo_flag', 'rango_antiguedad', 'rango_cargo_mensual', 'tipo_servicio', 'n_servicios_adicionales', 'rango_servicios_adicionales', 'pago_automatico', 'tiene_familia', 'cargo_mensual_perdido']


,antiguedad_meses,rango_antiguedad,cargo_mensual,rango_cargo_mensual,tipo_servicio,n_servicios_adicionales,pago_automatico,cancelo,cancelo_flag,cargo_mensual_perdido
0,1,0-12 meses,29.85,Bajo (<35),Solo internet,1,No,No,0,0.00
1,34,25-48 meses,56.95,Medio (35-70),Teléfono e internet,2,No,No,0,0.00
2,2,0-12 meses,53.85,Medio (35-70),Teléfono e internet,2,No,Sí,1,53.85
3,45,25-48 meses,42.30,Medio (35-70),Solo internet,3,Sí,No,0,0.00
4,2,0-12 meses,70.70,Alto (70-90),Teléfono e internet,0,No,Sí,1,70.70


In [8]:
# Validaciones del enriquecimiento
assert len(df_final) == len(df_original)                        # no se multiplicaron filas
assert df_final[nuevas].isnull().sum().sum() == 0               # todas las filas quedaron clasificadas
assert df_final["cancelo_flag"].sum() == (df_original["Churn"] == "Yes").sum()
assert round(df_final.loc[df_final.cancelo_flag == 1, "cargo_mensual"].sum(), 2) == round(df_final["cargo_mensual_perdido"].sum(), 2)
assert df_final["n_servicios_adicionales"].between(0, 6).all()
print("OK: enriquecimiento validado (sin nulos, sin filas extra, totales coherentes).")

OK: enriquecimiento validado (sin nulos, sin filas extra, totales coherentes).


## 5. Respuestas a las preguntas de negocio

Todas las tablas usan la misma estructura (función `tabla_tasa`):

| Columna | Significado | Unidad |
|---|---|---|
| `clientes` / `cancelaciones` | Clientes del segmento / clientes que cancelaron | clientes |
| `tasa_cancelacion_pct` | cancelaciones / clientes × 100 | % |
| `diferencia_vs_global_pp` | tasa del segmento − tasa global | puntos porcentuales |
| `participacion_clientes_pct` / `participacion_cancelaciones_pct` | % del total de clientes / de cancelaciones | % |
| `cargo_mensual_promedio` | promedio de `cargo_mensual` del segmento | u.m./mes |
| `cargo_mensual_perdido` / `participacion_cargo_perdido_pct` | suma del cargo mensual de quienes cancelaron / % del total perdido | u.m./mes, % |

Cada tabla se valida: por variable, los clientes suman 7.043, las cancelaciones 1.869 y las participaciones 100 %.

In [9]:
TABLAS = {}   # nombre de archivo -> tabla

def presentar(nombre, tabla, validar=True):
    if validar:
        validar_tabla(tabla, df_final)
    TABLAS[nombre] = tabla
    return tabla

### P0. Línea base: tasa global de cancelación

In [10]:
p0 = (df_final.groupby("cancelo", observed=True)
      .agg(clientes=("id_cliente", "count"), cargo_mensual_total=("cargo_mensual", "sum"),
           cargo_mensual_promedio=("cargo_mensual", "mean"), antiguedad_promedio_meses=("antiguedad_meses", "mean"))
      .reset_index())
p0["participacion_clientes_pct"] = p0["clientes"] / p0["clientes"].sum() * 100
p0["participacion_cargo_mensual_pct"] = p0["cargo_mensual_total"] / p0["cargo_mensual_total"].sum() * 100
p0 = p0.round(2)
assert p0["clientes"].sum() == len(df_final)
TASA_GLOBAL = df_final["cancelo_flag"].mean() * 100
TABLAS["P0_tasa_global"] = p0
print(f"Tasa global de cancelacion: {TASA_GLOBAL:.2f} %")
p0

Tasa global de cancelacion: 26.54 %


,cancelo,clientes,cargo_mensual_total,cargo_mensual_promedio,antiguedad_promedio_meses,participacion_clientes_pct,participacion_cargo_mensual_pct
0,No,5174,316985.75,61.27,37.57,73.46,69.5
1,Sí,1869,139130.85,74.44,17.98,26.54,30.5


**Respuesta P0:** 1.869 de 7.043 clientes cancelaron (**26,54 %**). Esos clientes representaban **139.130,85 u.m. al mes (30,50 % de la facturación mensual)**, porque pagaban más en promedio (74,44 vs. 61,27 u.m.). Esta tasa es la referencia para comparar los segmentos.

### P1. Condiciones de la cuenta

In [11]:
p1 = presentar("P1_tasa_por_cuenta",
               tabla_tasa(df_final, ["tipo_contrato", "metodo_pago", "factura_electronica", "pago_automatico"]))
p1

,variable,segmento,clientes,cancelaciones,tasa_cancelacion_pct,diferencia_vs_global_pp,participacion_clientes_pct,participacion_cancelaciones_pct,cargo_mensual_promedio,cargo_mensual_perdido,participacion_cargo_perdido_pct
0,tipo_contrato,Mes a mes,3875,1655,42.71,16.17,55.02,88.55,66.40,120847.10,86.86
1,tipo_contrato,Un año,1473,166,11.27,-15.27,20.91,8.88,65.05,14118.45,10.15
2,tipo_contrato,Dos años,1695,48,2.83,-23.71,24.07,2.57,60.77,4165.30,2.99
3,metodo_pago,Cheque electrónico,2365,1071,45.29,18.75,33.58,57.30,76.26,84288.75,60.58
4,metodo_pago,Cheque por correo,1612,308,19.11,-7.43,22.89,16.48,43.92,16803.60,12.08
5,metodo_pago,Tarjeta de crédito (automática),1522,232,15.24,-11.29,21.61,12.41,66.51,17946.60,12.90
6,metodo_pago,Transferencia bancaria (automática),1544,258,16.71,-9.83,21.92,13.80,67.19,20091.90,14.44
7,factura_electronica,No,2872,469,16.33,-10.21,40.78,25.09,51.99,29620.85,21.29
8,factura_electronica,Sí,4171,1400,33.57,7.03,59.22,74.91,73.55,109510.00,78.71
9,pago_automatico,No,3977,1379,34.67,8.14,56.47,73.78,63.15,101092.35,72.66


**Respuesta P1:** el contrato es la variable con mayor diferencia: **mes a mes cancela 42,71 %** frente a 11,27 % (un año) y 2,83 % (dos años), y concentra el **88,55 % de las cancelaciones**. Por método de pago destaca el **cheque electrónico (45,29 %)**; los pagos automáticos rondan 15-17 %. Con factura electrónica la tasa es 33,57 % frente a 16,33 % sin ella.

*Utilidad:* migrar clientes mes a mes a contratos más largos y promover el pago automático son focos claros. *Limitación:* no demuestra que el contrato cause la permanencia; puede reflejar clientes ya comprometidos.

### P2. Antigüedad

In [12]:
p2 = presentar("P2_tasa_por_antiguedad", tabla_tasa(df_final, "rango_antiguedad"))
p2

,variable,segmento,clientes,cancelaciones,tasa_cancelacion_pct,diferencia_vs_global_pp,participacion_clientes_pct,participacion_cancelaciones_pct,cargo_mensual_promedio,cargo_mensual_perdido,participacion_cargo_perdido_pct
0,rango_antiguedad,0-12 meses,2186,1037,47.44,20.90,31.04,55.48,56.10,68954.25,49.56
1,rango_antiguedad,13-24 meses,1024,294,28.71,2.17,14.54,15.73,61.36,23081.65,16.59
2,rango_antiguedad,25-48 meses,1594,325,20.39,-6.15,22.63,17.39,65.93,27462.50,19.74
3,rango_antiguedad,49-72 meses,2239,213,9.51,-17.02,31.79,11.40,73.95,19632.45,14.11


**Respuesta P2:** la tasa baja de forma constante con la antigüedad: **47,44 % en el primer año**, 28,71 % (13-24), 20,39 % (25-48) y 9,51 % (49-72 meses). El primer año concentra el **55,48 % de las cancelaciones** con solo el 31 % de los clientes.

*Utilidad:* el primer año es la ventana crítica para acciones de bienvenida y acompañamiento. *Limitación:* es un corte transversal; no se observa a los mismos clientes en el tiempo.

### P3. Servicios contratados

In [13]:
p3 = presentar("P3_tasa_por_servicios",
               tabla_tasa(df_final, ["servicio_internet", "tipo_servicio", "rango_servicios_adicionales",
                                     *ADICIONALES, "lineas_multiples"]))
p3

,variable,segmento,clientes,cancelaciones,tasa_cancelacion_pct,diferencia_vs_global_pp,participacion_clientes_pct,participacion_cancelaciones_pct,cargo_mensual_promedio,cargo_mensual_perdido,participacion_cargo_perdido_pct
0,servicio_internet,DSL,2421,459,18.96,-7.58,34.37,24.56,58.10,22529.20,16.19
1,servicio_internet,Fibra óptica,3096,1297,41.89,15.36,43.96,69.40,91.50,114300.05,82.15
2,servicio_internet,Sin internet,1526,113,7.40,-19.13,21.67,6.05,21.08,2301.60,1.65
3,tipo_servicio,Solo teléfono,1526,113,7.40,-19.13,21.67,6.05,21.08,2301.60,1.65
4,tipo_servicio,Solo internet,682,170,24.93,-1.61,9.68,9.10,42.03,6380.75,4.59
5,tipo_servicio,Teléfono e internet,4835,1586,32.80,6.27,68.65,84.86,81.76,130448.50,93.76
6,rango_servicios_adicionales,0,2219,475,21.41,-5.13,31.51,25.41,32.79,24516.85,17.62
7,rango_servicios_adicionales,1-2,1999,812,40.62,14.08,28.38,43.45,69.11,60281.75,43.33
8,rango_servicios_adicionales,3-4,1970,496,25.18,-1.36,27.97,26.54,83.42,45499.10,32.70
9,rango_servicios_adicionales,5-6,855,86,10.06,-16.48,12.14,4.60,94.56,8833.15,6.35


**Respuesta P3:** los clientes con **fibra óptica cancelan 41,89 %** (DSL 18,96 %, sin internet 7,40 %) y aportan el **82,15 % del ingreso mensual perdido**. Quienes **no tienen seguridad en línea (41,77 %) o soporte técnico (41,64 %)** cancelan casi el triple que quienes sí los tienen (14,61 % y 15,17 %). Por número de adicionales, la tasa es mayor con 1-2 servicios (40,62 %) y mínima con 5-6 (10,06 %). Streaming y líneas múltiples casi no cambian la tasa.

*Utilidad:* ofrecer seguridad y soporte a clientes de fibra sin esos servicios. *Limitación:* no hay datos de calidad del servicio ni de precios por producto para explicar la alta cancelación en fibra.

### P4. Perfil demográfico

In [14]:
p4 = presentar("P4_tasa_por_perfil",
               tabla_tasa(df_final, ["genero", "adulto_mayor", "tiene_pareja", "tiene_dependientes", "tiene_familia"]))
p4

,variable,segmento,clientes,cancelaciones,tasa_cancelacion_pct,diferencia_vs_global_pp,participacion_clientes_pct,participacion_cancelaciones_pct,cargo_mensual_promedio,cargo_mensual_perdido,participacion_cargo_perdido_pct
0,genero,Femenino,3488,939,26.92,0.38,49.52,50.24,65.20,70248.55,50.49
1,genero,Masculino,3555,930,26.16,-0.38,50.48,49.76,64.33,68882.30,49.51
2,adulto_mayor,No,5901,1393,23.61,-2.93,83.79,74.53,61.85,100711.25,72.39
3,adulto_mayor,Sí,1142,476,41.68,15.14,16.21,25.47,79.82,38419.60,27.61
4,tiene_pareja,No,3641,1200,32.96,6.42,51.70,64.21,61.95,85741.15,61.63
5,tiene_pareja,Sí,3402,669,19.66,-6.87,48.30,35.79,67.78,53389.70,38.37
6,tiene_dependientes,No,4933,1543,31.28,4.74,70.04,82.56,67.00,115376.50,82.93
7,tiene_dependientes,Sí,2110,326,15.45,-11.09,29.96,17.44,59.52,23754.35,17.07
8,tiene_familia,No,3280,1123,34.24,7.70,46.57,60.09,62.98,80705.65,58.01
9,tiene_familia,Sí,3763,746,19.82,-6.71,53.43,39.91,66.31,58425.20,41.99


**Respuesta P4:** el género no marca diferencia (26,92 % vs. 26,16 %). Los **adultos mayores cancelan 41,68 %** (vs. 23,61 %), y quienes **no tienen pareja ni dependientes** cancelan 34,24 % frente a 19,82 % de quienes tienen familia.

*Utilidad:* adultos mayores y clientes sin familia son grupos a priorizar. *Limitación:* la edad que define "adulto mayor" no está documentada.

### P5. Cargos de quienes cancelan y quienes permanecen

In [15]:
p5a = (df_final.groupby("cancelo", observed=True)[["cargo_mensual", "cargo_total", "antiguedad_meses"]]
       .describe().T.round(2).reset_index()
       .rename(columns={"level_0": "variable", "level_1": "estadistica"}))
p5a["unidad"] = p5a["variable"].map({"cargo_mensual": "u.m./mes", "cargo_total": "u.m.", "antiguedad_meses": "meses"})
assert p5a.query("estadistica == 'count'")[["No", "Sí"]].sum(axis=1).eq(len(df_final)).all()
TABLAS["P5a_estadisticas_cargos"] = p5a
p5a

cancelo,variable,estadistica,No,Sí,unidad
0,cargo_mensual,count,5174.00,1869.00,u.m./mes
1,cargo_mensual,mean,61.27,74.44,u.m./mes
2,cargo_mensual,std,31.09,24.67,u.m./mes
3,cargo_mensual,min,18.25,18.85,u.m./mes
4,cargo_mensual,25%,25.10,56.15,u.m./mes
5,cargo_mensual,50%,64.43,79.65,u.m./mes
6,cargo_mensual,75%,88.40,94.20,u.m./mes
7,cargo_mensual,max,118.75,118.35,u.m./mes
8,cargo_total,count,5174.00,1869.00,u.m.
9,cargo_total,mean,2549.91,1531.80,u.m.


In [16]:
p5b = presentar("P5b_tasa_por_rango_cargo", tabla_tasa(df_final, "rango_cargo_mensual"))
p5b

,variable,segmento,clientes,cancelaciones,tasa_cancelacion_pct,diferencia_vs_global_pp,participacion_clientes_pct,participacion_cancelaciones_pct,cargo_mensual_promedio,cargo_mensual_perdido,participacion_cargo_perdido_pct
0,rango_cargo_mensual,Bajo (<35),1731,188,10.86,-15.68,24.58,10.06,22.00,4423.05,3.18
1,rango_cargo_mensual,Medio (35-70),1721,407,23.65,-2.89,24.44,21.78,54.72,21726.80,15.62
2,rango_cargo_mensual,Alto (70-90),1847,701,37.95,11.42,26.22,37.51,79.99,55926.90,40.20
3,rango_cargo_mensual,Muy alto (>=90),1744,573,32.86,6.32,24.76,30.66,100.99,57054.10,41.01


**Respuesta P5:** quienes cancelan pagan **más al mes** (media 74,44 vs. 61,27 u.m.; mediana 79,65 vs. 64,43) pero acumulan **menos cargo total** (mediana 703,55 vs. 1.679,52 u.m.), porque llevan menos tiempo (mediana 10 vs. 38 meses). La tasa sube de 10,86 % en cargo bajo (<35) a **37,95 % en cargo alto (70-90)** y 32,86 % en muy alto (≥90); estos dos rangos suman el **81 % del ingreso mensual perdido**.

*Utilidad:* el riesgo se concentra en clientes nuevos de cargo alto. *Limitación:* sin costos no se puede hablar de rentabilidad; moneda no documentada.

### P6. Segmentos combinados con mayor cancelación

Se combinan las tres variables con mayor diferencia (contrato × internet × antigüedad). Para evitar tasas engañosas en grupos pequeños se exigen **al menos 50 clientes** por segmento.

In [17]:
MIN_CLIENTES = 50
df_seg = df_final.assign(segmento_combinado=df_final["tipo_contrato"].astype(str) + " | "
                         + df_final["servicio_internet"].astype(str) + " | "
                         + df_final["rango_antiguedad"].astype(str))
p6_todos = tabla_tasa(df_seg, "segmento_combinado", total_df=df_final)
validar_tabla(p6_todos, df_final)    # todos los segmentos juntos suman el total
p6 = (p6_todos[p6_todos["clientes"] >= MIN_CLIENTES]
      .sort_values("tasa_cancelacion_pct", ascending=False).head(10).reset_index(drop=True))
p6.insert(0, "ranking", range(1, len(p6) + 1))
p6[["tipo_contrato", "servicio_internet", "rango_antiguedad"]] = p6["segmento"].str.split(" \| ", expand=True)
TABLAS["P6_top10_segmentos"] = p6
print("Segmentos posibles:", len(p6_todos), "| con >= 50 clientes:", (p6_todos.clientes >= MIN_CLIENTES).sum())
p6

Segmentos posibles: 35 | con >= 50 clientes: 27


,ranking,variable,segmento,clientes,cancelaciones,tasa_cancelacion_pct,diferencia_vs_global_pp,participacion_clientes_pct,participacion_cancelaciones_pct,cargo_mensual_promedio,cargo_mensual_perdido,participacion_cargo_perdido_pct,tipo_contrato,servicio_internet,rango_antiguedad
0,1,segmento_combinado,Mes a mes | Fibra óptica | 0-12 meses,916,643,70.20,43.66,13.01,34.40,82.08,53178.30,38.22,Mes a mes,Fibra óptica,0-12 meses
1,2,segmento_combinado,Mes a mes | Fibra óptica | 13-24 meses,425,215,50.59,24.05,6.03,11.50,87.54,19046.90,13.69,Mes a mes,Fibra óptica,13-24 meses
2,3,segmento_combinado,Mes a mes | Fibra óptica | 25-48 meses,521,226,43.38,16.84,7.40,12.09,91.24,20762.05,14.92,Mes a mes,Fibra óptica,25-48 meses
3,4,segmento_combinado,Mes a mes | DSL | 0-12 meses,690,293,42.46,15.93,9.80,15.68,47.88,13351.90,9.60,Mes a mes,DSL,0-12 meses
4,5,segmento_combinado,Mes a mes | Fibra óptica | 49-72 meses,266,78,29.32,2.79,3.78,4.17,94.95,7494.75,5.39,Mes a mes,Fibra óptica,49-72 meses
5,6,segmento_combinado,Mes a mes | DSL | 13-24 meses,232,55,23.71,-2.83,3.29,2.94,52.68,2765.40,1.99,Mes a mes,DSL,13-24 meses
6,7,segmento_combinado,Mes a mes | Sin internet | 0-12 meses,388,88,22.68,-3.86,5.51,4.71,20.27,1771.25,1.27,Mes a mes,Sin internet,0-12 meses
7,8,segmento_combinado,Un año | Fibra óptica | 25-48 meses,154,31,20.13,-6.41,2.19,1.66,96.04,3039.10,2.18,Un año,Fibra óptica,25-48 meses
8,9,segmento_combinado,Un año | Fibra óptica | 49-72 meses,355,67,18.87,-7.66,5.04,3.58,100.49,6926.20,4.98,Un año,Fibra óptica,49-72 meses
9,10,segmento_combinado,Mes a mes | DSL | 25-48 meses,227,36,15.86,-10.68,3.22,1.93,53.88,1755.75,1.26,Mes a mes,DSL,25-48 meses


In [18]:
top3 = p6.head(3)
print(f"Top 3 segmentos: {top3.clientes.sum()} clientes ({top3.participacion_clientes_pct.sum():.2f} %), "
      f"{top3.cancelaciones.sum()} cancelaciones ({top3.participacion_cancelaciones_pct.sum():.2f} %), "
      f"{top3.cargo_mensual_perdido.sum():,.2f} u.m./mes perdidos ({top3.participacion_cargo_perdido_pct.sum():.2f} %)")

Top 3 segmentos: 1862 clientes (26.44 %), 1084 cancelaciones (57.99 %), 92,987.25 u.m./mes perdidos (66.83 %)


**Respuesta P6:** el segmento más crítico es **mes a mes + fibra óptica + 0-12 meses: 70,20 % de cancelación** (916 clientes, 34,40 % de todas las cancelaciones). Los tres primeros segmentos son todos *mes a mes + fibra* y reúnen cerca del 26 % de los clientes pero alrededor del **58 % de las cancelaciones y el 67 % del ingreso mensual perdido**.

*Utilidad:* es la lista concreta de segmentos a priorizar. *Limitación:* combinar más variables reduce el tamaño de los grupos; por eso se fijó el mínimo de 50 clientes.

## 6. Exportación de resultados (rutas relativas)

In [19]:
DIR_DATOS = AE3 / "datos"
DIR_TABLAS = AE3 / "resultados" / "tablas"
DIR_DATOS.mkdir(parents=True, exist_ok=True)
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

RUTA_DATASET = DIR_DATOS / "telco_churn_preparado_Actividad4.csv"
df_final.to_csv(RUTA_DATASET, index=False, encoding="utf-8")

# Tablas de calidad (evidencia de la transformacion)
TABLAS["calidad_registro_transformaciones"] = registro_cambios
TABLAS["calidad_controles"] = controles
TABLAS["calidad_comparacion_antes_despues"] = comparacion.reset_index()

for nombre, tabla in TABLAS.items():
    tabla.to_csv(DIR_TABLAS / f"{nombre}_Actividad4.csv", index=False, encoding="utf-8")

for ruta in [RUTA_DATASET, *sorted(DIR_TABLAS.glob("*_Actividad4.csv"))]:
    print(ruta.relative_to(RAIZ).as_posix())

AE3/datos/telco_churn_preparado_Actividad4.csv


AE3/resultados/tablas/calidad_comparacion_antes_despues_Actividad4.csv
AE3/resultados/tablas/calidad_controles_Actividad4.csv
AE3/resultados/tablas/calidad_registro_transformaciones_Actividad4.csv
AE3/resultados/tablas/P0_tasa_global_Actividad4.csv
AE3/resultados/tablas/P1_tasa_por_cuenta_Actividad4.csv
AE3/resultados/tablas/P2_tasa_por_antiguedad_Actividad4.csv
AE3/resultados/tablas/P3_tasa_por_servicios_Actividad4.csv
AE3/resultados/tablas/P4_tasa_por_perfil_Actividad4.csv
AE3/resultados/tablas/P5a_estadisticas_cargos_Actividad4.csv
AE3/resultados/tablas/P5b_tasa_por_rango_cargo_Actividad4.csv
AE3/resultados/tablas/P6_top10_segmentos_Actividad4.csv


### 6.1 Verificación: los archivos exportados coinciden con el notebook

In [20]:
leido = pd.read_csv(RUTA_DATASET)
assert leido.shape == df_final.shape
assert leido["cancelo_flag"].sum() == df_final["cancelo_flag"].sum()
assert abs(leido["cargo_mensual_perdido"].sum() - df_final["cargo_mensual_perdido"].sum()) < 0.01
for nombre, tabla in TABLAS.items():
    csv = pd.read_csv(DIR_TABLAS / f"{nombre}_Actividad4.csv")
    assert csv.shape == tabla.shape, nombre
    for col in tabla.select_dtypes("number").columns:
        assert abs(csv[col].sum() - tabla[col].sum()) < 0.01, (nombre, col)
print(f"OK: dataset ({leido.shape[0]} x {leido.shape[1]}) y {len(TABLAS)} tablas coinciden con el notebook.")

OK: dataset (7043 x 30) y 11 tablas coinciden con el notebook.


## 7. Resumen descriptivo automático (.txt) del DataFrame final

In [21]:
RUTA_TXT = AE3 / "resultados" / "resumen_dataframe_Actividad4.txt"
procedencia = {
    "Dataset": "Telco Customer Churn (IBM Sample Dataset)",
    "Identificador": DATASET_ID,
    "Revision": REVISION,
    "Licencia": "CC BY 4.0 (permite redistribuir citando la fuente)",
    "Estado": "Transformado y enriquecido (AE3 - Actividad 4)",
    "Muestra": "No - incluye los 7.043 clientes",
    "Dataset exportado": RUTA_DATASET.relative_to(RAIZ).as_posix(),
}
ruta_txt = ResumenDescriptivo(df_final, "Resumen del DataFrame final - Telco Churn (AE3)",
                              procedencia, registro_cambios).guardar(RUTA_TXT)

contenido = ruta_txt.read_text(encoding="utf-8")
assert f"Filas: {df_final.shape[0]}" in contenido and f"Columnas: {df_final.shape[1]}" in contenido
assert all(c in contenido for c in df_final.columns)
assert f"{TASA_GLOBAL:.2f} %" in contenido
print("Resumen regenerado y verificado:", ruta_txt.relative_to(RAIZ).as_posix(), "\n")
print(contenido[:3000], "\n...")

Resumen regenerado y verificado: AE3/resultados/resumen_dataframe_Actividad4.txt 

RESUMEN DEL DATAFRAME FINAL - TELCO CHURN (AE3)
Generado: 2026-11-08 18:34:57

1. Procedencia
--------------
Dataset           : Telco Customer Churn (IBM Sample Dataset)
Identificador     : scikit-learn/churn-prediction
Revision          : aa09900373d90780ee70d27571775aff0e51569c
Licencia          : CC BY 4.0 (permite redistribuir citando la fuente)
Estado            : Transformado y enriquecido (AE3 - Actividad 4)
Muestra           : No - incluye los 7.043 clientes
Dataset exportado : AE3/datos/telco_churn_preparado_Actividad4.csv

2. Dimensiones
--------------
Filas: 7043
Columnas: 30

3. Columnas y tipos de datos
----------------------------
                                 tipo  valores_unicos
id_cliente                     string            7043
genero                       category               2
adulto_mayor                 category               2
tiene_pareja                 category          

## 8. Conclusiones del avance

- Se transformaron los 7.043 clientes sin eliminar registros: `TotalCharges` pasó a número (11 vacíos → 0 por ser clientes con antigüedad 0), se normalizaron categorías y nombres, y todos los controles de calidad resultaron OK.
- Se crearon 9 variables (rangos, tipo de servicio, conteo de adicionales, pago automático, familia, indicador de cancelación e ingreso perdido) que permitieron responder las 6 preguntas.
- **Hallazgo central:** la cancelación (26,54 % global) se concentra en clientes **mes a mes, con fibra óptica, en su primer año**, sin seguridad ni soporte técnico, que pagan con cheque electrónico y tienen cargos altos.
- **Sin resolver:** causas de la cancelación, costos y rentabilidad, evolución en el tiempo y moneda; requieren otros datos.
- Las tablas `.csv` en `AE3/resultados/tablas/` quedan listas para las gráficas de AE4.